In [1]:
import pandas as pd

### Попытки мерджа двух датасетов - временные ряды и табличные данные

**Зачем** - датасет с табличными данными содержит основную информацию о пациенте, исход и временные рамки лечения. Это потребуется для создания модели прогноза временных рядов. 
Так же, в будущем табличный датасет может быть обогащен информацией от временных рядов для лучшей классификации исхода. 

In [2]:
table_data = pd.read_excel("./data_raw/DataSet_V49 (2).xlsx")

In [3]:
time_series_data = pd.read_excel("./data_raw/data-timeseries_test.xlsx")

In [4]:
time_series_data["chartTime"] = pd.to_datetime(time_series_data["chartTime"])

In [42]:
print(table_data[table_data["Name"].str.contains("Билан", na=True)].to_markdown())

|       | Код пациента   | Name     |   Age | Sex   | Наличие в БД   | Наличие в файле   | STEMI   | ЧКВ   |   Дата STEMI |   Вид STEMI |   SYNTAX Score |   Инфаркт-зависимая артерия | Поражение ствола   |   Количество пораженных сосудов(Syntax) |   Количество пораженных сосудов(Значимость) |   TIMI | Инфаркт миокарда в анамнезе (<3)   | Инфаркт миокарда в анамнезе (>3)   |   Инфаркт миокарда со стентированием в анамнезе |   ОНМК (иш) в анамнезе |   ОНМК (гем) в анамнезе |   Стентирование в анамнезе |   Тромболизис |   Форма ФП |   Калий |   Дилатация предсердий | ФП b (после чкв)   | ФП a (в анамнезе)   |   ФП при окс (до чкв) |   ФП постоянная форма |   ФЖ |   Пробежки ЖТ |   Рецидивирующая ЖТ |   MKB |   Класс ОСН по Killip | ХСН стадия   | ХСН фк   |   ГБ стадия |   ГБ риск | АГ   | Стенокардия (ИБ)   | Стенокардия форма(ИБ)   |   Стенокардия ФК (ИБ) | СД   | ХБП   | ЯБ   | МКБ   |   ЖКБ |   ФВ ЛЖ (b) |   ФВ ЛЖ |   EDV |   КДР ЛЖ |   КСР ЛЖ |   МЖП |   ЗСЛЖ |   ПСПЖ |   СДЛА |   Ра

In [242]:
import re

def extract_year_id(value):
    """
    Возвращает кортеж (год, идентификатор) или (None, идентификатор) или None.
    Год преобразуется в полный (2000+год).
    """
    if not isinstance(value, str):
        value = str(value)
    value = value.strip()

    # Паттерн: две цифры, дефис, одна и более цифр (год-ид)
    match1 = re.fullmatch(r'^(\d{2})-(\d+)$', value)
    if match1:
        year = int(match1.group(1))
        patient_id = str(match1.group(2))
        return (2000 + year, patient_id)

    # Паттерн: одна и более цифр, дефис, две цифры (ид-год)
    match2 = re.fullmatch(r'^(\d+)-(\d{2})$', value)
    if match2:
        patient_id = str(match2.group(1))
        year = int(match2.group(2))
        return (2000 + year, patient_id)

    # Только цифры (без дефиса) — идентификатор, год неизвестен
    if value.isdigit():
        patient_id = str(value)
        return (None, patient_id)

    # Не подходит ни под один формат → отбрасываем
    return None

In [243]:
table_data[['year', 'patient_id']] = table_data['Код пациента'].apply(
    lambda x: pd.Series(extract_year_id(x))
)

In [240]:
table_data["patient_id"] = table_data["patient_id"].astype("str")

In [244]:
table_data

,Код пациента,Name,Age,Sex,Наличие в БД,Наличие в файле,STEMI,ЧКВ,Дата STEMI,Вид STEMI,...,HCO3VenMin (a),HCO3VenMin,HCO3VenMax (b),HCO3VenMax (a),HCO3VenMax,BNP (b),BNP (a),BNP,year,patient_id
0,20-6545,Хасьянов РА,78.0,М,Да,Да,Нет,Да,NaN,NaN,...,18.5,18.5,18.5,18.5,18.5,NaN,NaN,NaN,2020.0,6545
1,19-22109,Тюркова ГГ,80.0,Ж,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.0,22109
2,17-13439,Ремизов РВ,40.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017.0,13439
3,16-20326,Синенко ОП,78.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016.0,20326
4,9993,Комолов ВИ,75.0,М,Да,Да,Да,Да,2015-07-09 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9993
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17425,10019,Гаевский АИ,60.0,М,Да,Да,Да,Да,2021-06-08 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10019
17426,10,Сальникова ВС,81.0,Ж,Да,Да,Да,Да,2020-12-31 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10
17427,1,Халиман ОА,48.0,М,Да,Да,Да,Да,2020-01-01 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
17428,2024-06-18 00:00:00,Журавлев АЮ,50.0,М,Да,Да,Да,Да,2018-01-01 00:00:00,Задний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Загрузка

Загружаем все датафреймы в виде отдельных файлов для дальнейшей фильтрации

In [55]:
time_2020 = pd.read_excel("./data_raw/time_series/2020.xlsx")
time_2021 = pd.read_excel("./data_raw/time_series/2021.xlsx")
time_2022 = pd.read_excel("./data_raw/time_series/2022.xlsx")
time_2023 = pd.read_excel("./data_raw/time_series/2023.xlsx")
time_2024 = pd.read_excel("./data_raw/time_series/2024.xlsx")
time_2025 = pd.read_excel("./data_raw/time_series/2025.xlsx")

In [73]:
time_2023

,№иб,ФИО,Возраст,Диагноз,ХЗ,ХЗ1,ХЗ2,ХЗ3,ХЗ4,ХЗ5,ХЗ6,ХЗ7,ХЗ8,ХЗ9,ХЗ10,ХЗ11
0,23- 10341,Сапко АВ,67,нестабильная,КО,False,True,45067,45068,False,NaN,2,владивосток,NaN,Сахарный диабет,ХСН
1,23-1,Петрунькин ИИ,58,nstemi передний,КО,False,True,44927,44929,False,ДН,3,Владивосток,NaN,ХСН,NaN
2,23-10,Бараненко АА,83,stemi боковой,КО,False,True,44928,44929,False,NaN,2,Владивосток,NaN,ХСН,NaN
3,23-10002,Зубова ИН,60,нестабильная,КО,False,True,45062,45064,False,NaN,3,прим край,"ХОБЛ, ЯБДПК",Сахарный диабет,6 и др
4,23-10007,Вербианов СВ,61,STEMI передний,КО,False,True,45062,45063,False,NaN,2,прим край,NaN,Сахарный диабет,ХСН
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1944,23-9905,Коновал БФ,75,STEMI задний,КО,False,True,45062,45063,False,острая декомпенсация ХСН,2,владивосток,NaN,ХСН,ФП постоянная форма
1945,23-9939,Банах ВЕ,61,"Госпитальная пневмония, состояние после АКШ",КХО,False,True,45074,45075,False,NaN,2,владивосток,NaN,ХОБЛ,NaN
1946,23-9939,Банах ВЕ,61,"Госпитальная пневмония, состояние после АКШ",КХО,True,True,45078,45086,True,"ОДН, острая декомпенсация ХСН, сепсис",9,владивосток,ИВЛ 9 суток,ХОБЛ,NaN
1947,23-9976,Симоненко АП,83,ФП,ОНР,False,True,45062,45063,False,острая декомпенсация ХСН,2,прим край,делирий,Сахарный диабет,ХСН


### Собираем

Каждый файл обрабатывается отдельно. Не самое удобное решение, стоит переписать. 

В табличных данных есть колонка **"Код пациента"**, она связывается с **"№иб"** из файлов с временными рядами. 
Иногда, совпадение может быть неточным и мы полагаемся на то, что связь будет похожа. Стоит так же проверять по ФИО, но там так же нет точных совпадений. 

После фильтрации, мы накапливаем отфильтрованные строки и собираем в единый датафрейм. 

Так мы делаем с каждым файлом, получая строки из табличного датасета

In [245]:
result_dfs_2020 = []

for val in time_2020["№иб"].values:    
    # filtered = table_data[table_data["Код пациента"].str.contains(val, na=False, regex=False)]
    filtered = table_data[table_data["Код пациента"] == val]
    if not filtered.empty:
        filtered = filtered.copy()
        filtered["match"] = val 
        result_dfs_2020.append(filtered)

if result_dfs_2020:
    final_df_2020 = pd.concat(result_dfs_2020, ignore_index=True)
    print("Done!")
else:
    print("Ничего не найдено")

Done!


In [246]:
result_dfs_2021 = []

for val in time_2021["№иб"].values:    
    # filtered = table_data[table_data["Код пациента"].str.contains(val, na=False, regex=False)]
    filtered = table_data[table_data["Код пациента"] == val]
    if not filtered.empty:
        filtered = filtered.copy()
        filtered["match"] = val 
        result_dfs_2021.append(filtered)

if result_dfs_2021:
    final_df_2021 = pd.concat(result_dfs_2021, ignore_index=True)
    print("Done!")
else:
    print("Ничего не найдено")

Done!


In [247]:
result_dfs_2022 = []

for val in time_2022["№иб"].values:    
    # filtered = table_data[table_data["Код пациента"].str.contains(val, na=False, regex=False)]
    filtered = table_data[table_data["Код пациента"] == val]
    if not filtered.empty:
        filtered = filtered.copy()
        filtered["match"] = val 
        result_dfs_2022.append(filtered)

if result_dfs_2022:
    final_df_2022 = pd.concat(result_dfs_2022, ignore_index=True)
    print("Done!")
else:
    print("Ничего не найдено")

Done!


In [248]:
result_dfs_2023 = []

for val in time_2023["№иб"].values:    
    # filtered = table_data[table_data["Код пациента"].str.contains(val, na=False, regex=False)]
    filtered = table_data[table_data["Код пациента"] == val]
    if not filtered.empty:
        filtered = filtered.copy()
        filtered["match"] = val 
        result_dfs_2023.append(filtered)

if result_dfs_2023:
    final_df_2023 = pd.concat(result_dfs_2023, ignore_index=True)
    print("Done!")
else:
    print("Ничего не найдено")

Done!


In [249]:
result_dfs_2024 = []

for val in time_2024["№иб"].values:    
    # filtered = table_data[table_data["Код пациента"].str.contains(val, na=False, regex=False)]
    filtered = table_data[table_data["Код пациента"] == val]
    if not filtered.empty:
        filtered = filtered.copy()
        filtered["match"] = val 
        result_dfs_2024.append(filtered)

if result_dfs_2024:
    final_df_2024 = pd.concat(result_dfs_2024, ignore_index=True)
    print("Done!")
else:
    print("Ничего не найдено")

Done!


### Для файла с 2025 годом нет никаких совпадений!

In [200]:
result_dfs_2025 = []

for val in time_2025["№иб"].values:    
    filtered = table_data[table_data["Код пациента"] == val]
    # filtered = table_data[table_data["Код пациента"].str.contains(val, na=False, regex=False)]
    if not filtered.empty:
        filtered = filtered.copy()
        filtered["match"] = val 
        result_dfs_2025.append(filtered)

if result_dfs_2025:
    final_df_2025 = pd.concat(result_dfs_2025, ignore_index=True)
    print("Done!")
else:
    print("Ничего не найдено")

Ничего не найдено


In [201]:
print(f"Количество строк в датасете 2020: {len(final_df_2020)}")
print(f"Количество строк в датасете 2021: {len(final_df_2021)}")
print(f"Количество строк в датасете 2022: {len(final_df_2022)}")
print(f"Количество строк в датасете 2023: {len(final_df_2023)}")
print(f"Количество строк в датасете 2024: {len(final_df_2024)}")

Количество строк в датасете 2020: 767
Количество строк в датасете 2021: 1479
Количество строк в датасете 2022: 620
Количество строк в датасете 2023: 726
Количество строк в датасете 2024: 1284


### Собираем

Собираем все полученные отфильтрованные строки в единый датафрейм

In [251]:
mergerd_df = pd.concat([final_df_2020, final_df_2021, final_df_2022, final_df_2023, final_df_2024], ignore_index=True)

In [203]:
mergerd_df

,Код пациента,Name,Age,Sex,Наличие в БД,Наличие в файле,STEMI,ЧКВ,Дата STEMI,Вид STEMI,...,HCO3VenMin (b),HCO3VenMin (a),HCO3VenMin,HCO3VenMax (b),HCO3VenMax (a),HCO3VenMax,BNP (b),BNP (a),BNP,match
0,20-10653,Каменева ТЛ,81.0,Ж,Нет,Да,Нет,Нет,NaN,NaN,...,25.1,25.1,25.1,25.1,25.1,25.1,NaN,NaN,NaN,20-10653
1,20-10653,Каменева ТЛ,81.0,Ж,Нет,Да,Нет,Нет,NaN,NaN,...,25.1,25.1,25.1,25.1,25.1,25.1,NaN,NaN,NaN,20-10653
2,20-10654,Заболотная ТГ,81.0,Ж,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20-10654
3,20-10654,Заболотная ТГ,81.0,Ж,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20-10654
4,20-10656,Абдразяков ОЖ,53.0,М,Да,Да,Да,Да,2020-07-15 00:00:00,Передний,...,25.4,25.4,25.4,25.4,25.4,25.4,NaN,NaN,NaN,20-10656
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4871,24-9663,Перова КД,85.0,Ж,Да,Да,Да,Да,2024-05-15 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24-9663
4872,24-9775,Антонов СН,45.0,М,Да,Да,Да,Да,2024-05-13 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24-9775
4873,24-9805,Пщебильский АВ,65.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24-9805
4874,24-9906,Игумнов ВМ,75.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,818.2,818.2,24-9906


In [252]:
mergerd_df[['year', 'patient_id']] = mergerd_df['Код пациента'].apply(
    lambda x: pd.Series(extract_year_id(x))
)

In [253]:
mergerd_df

,Код пациента,Name,Age,Sex,Наличие в БД,Наличие в файле,STEMI,ЧКВ,Дата STEMI,Вид STEMI,...,HCO3VenMin,HCO3VenMax (b),HCO3VenMax (a),HCO3VenMax,BNP (b),BNP (a),BNP,year,patient_id,match
0,20-10653,Каменева ТЛ,81.0,Ж,Нет,Да,Нет,Нет,NaN,NaN,...,25.1,25.1,25.1,25.1,NaN,NaN,NaN,2020,10653,20-10653
1,20-10653,Каменева ТЛ,81.0,Ж,Нет,Да,Нет,Нет,NaN,NaN,...,25.1,25.1,25.1,25.1,NaN,NaN,NaN,2020,10653,20-10653
2,20-10654,Заболотная ТГ,81.0,Ж,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020,10654,20-10654
3,20-10654,Заболотная ТГ,81.0,Ж,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020,10654,20-10654
4,20-10656,Абдразяков ОЖ,53.0,М,Да,Да,Да,Да,2020-07-15 00:00:00,Передний,...,25.4,25.4,25.4,25.4,NaN,NaN,NaN,2020,10656,20-10656
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4871,24-9663,Перова КД,85.0,Ж,Да,Да,Да,Да,2024-05-15 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,9663,24-9663
4872,24-9775,Антонов СН,45.0,М,Да,Да,Да,Да,2024-05-13 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,9775,24-9775
4873,24-9805,Пщебильский АВ,65.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,9805,24-9805
4874,24-9906,Игумнов ВМ,75.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,818.2,818.2,2024,9906,24-9906


In [204]:
for i in list(mergerd_df.columns):
    if "Дата" in i or "дата" in i:
        print(i)

Дата STEMI
Дата направления (Общий анализ крови)
Дата взятия биоматериала (Общий анализ крови)
Дата выполнения (Общий анализ крови)
Дата направления (Общий анализ крови-экспрес)
Дата взятия биоматериала (Общий анализ крови-экспрес)
Дата выполнения (Общий анализ крови-экспрес)
дата поступления
дата выписки
дата смерти
Дата и время развития SOFA 8 и более


In [205]:
time_2025

,№иб,ФИО,Возраст,Диагноз,ХЗ,ХЗ1,ХЗ2,ХЗ3,ХЗ4,ХЗ5,ХЗ6,ХЗ7,ХЗ8,ХЗ9,ХЗ10,ХЗ11
0,25-1,Руденок ВН,71,stemi передний,КО,False,True,45658,45659.0,False,NaN,2.0,Владивосток,NaN,NaN,NaN
1,25-10,Дроздов АИ,65,stemi задний,КО,False,True,45658,45659.0,False,NaN,2.0,Владивосток,NaN,NaN,NaN
2,25-1000,Несветов КН,60,STEMI задний,КО,False,True,45687,45688.0,False,NaN,2.0,прим край,NaN,"ЯБЖ, ЯБДПК",NaN
3,25-1023,Бамбурова НГ,59,STEMI задний,КО,False,True,45688,45690.0,False,ОССН,3.0,прим край,NaN,NaN,NaN
4,25-1025,Иньков ОЮ,53,"нестабильная, АКШ",КХО,False,True,45688,45689.0,False,NaN,2.0,прим край,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1625,25-97,Батурин АВ,50,нестабильная,КО,False,True,45663,45664.0,False,NaN,2.0,прим край,NaN,NaN,NaN
1626,25-99,Медведчук АА,83,nSTEMI,КО,False,True,45663,45665.0,False,NaN,3.0,владивосток,NaN,Сахарный диабет,ХСН
1627,25-993,Басов СА,52,STEMI задний,КО,False,True,45687,45688.0,False,NaN,2.0,прим край,NaN,NaN,NaN
1628,25-997,Мельчакова ОС,71,нестабильная,КО,False,True,45687,45688.0,False,NaN,2.0,прим край,NaN,Сахарный диабет,NaN


In [254]:
mergerd_df["дата выписки"] = pd.to_datetime(mergerd_df["дата выписки"], errors='coerce')
mergerd_df["дата поступления"] = pd.to_datetime(mergerd_df["дата поступления"], errors='coerce')
mergerd_df = mergerd_df.sort_values("дата поступления")

In [207]:
mergerd_df

,Код пациента,Name,Age,Sex,Наличие в БД,Наличие в файле,STEMI,ЧКВ,Дата STEMI,Вид STEMI,...,HCO3VenMin (b),HCO3VenMin (a),HCO3VenMin,HCO3VenMax (b),HCO3VenMax (a),HCO3VenMax,BNP (b),BNP (a),BNP,match
5,20-10689,Панченко ЛН,59.0,М,Да,Да,Нет,Да,NaN,NaN,...,23.7,23.7,23.7,23.7,23.7,23.7,NaN,NaN,NaN,20-10689
45,20-11028,Марченко ВА,66.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20-11028
59,20-11164,Подленный КИ,73.0,М,Да,Да,Нет,Да,NaN,NaN,...,21.1,21.1,21.1,21.1,21.1,21.1,NaN,NaN,NaN,20-11164
68,20-11216,Калинин АИ,73.0,М,Да,Да,Нет,Да,NaN,NaN,...,25.0,25.0,25.0,25.0,25.0,25.0,NaN,NaN,NaN,20-11216
69,20-11220,Волков АА,47.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20-11220
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2825,22-858,Мирошников КВ,52.0,М,Да,Да,Да,Да,2022-01-16 00:00:00,Задний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22-858
2839,22-910,Родин ИВ,49.0,М,Да,Да,Да,Да,2021-01-16 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22-910
2850,22-947,Смирнова НН,70.0,Ж,Да,Да,Да,Да,2022-01-16 00:00:00,Задний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22-947
2853,22-962,Тимохин СН,50.0,М,Да,Да,Да,Да,2022-01-17 00:00:00,Задний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22-962


In [208]:
time_series_data = time_series_data.sort_values("chartTime")

In [209]:
mergerd_df.rename(columns={"Код пациента": "patient_id"}, inplace=True)
time_series_data.rename(columns={"№иб": "patient_id"}, inplace=True)
time_series_data.rename(columns={"N иб": "patient_id"}, inplace=True)

In [210]:
time_series_data["patient_id"] = time_series_data["patient_id"].astype('object')

In [211]:
time_series_data["patient_id"].info()

<class 'pandas.core.series.Series'>
Index: 44246 entries, 0 to 43129
Series name: patient_id
Non-Null Count  Dtype 
--------------  ----- 
44246 non-null  object
dtypes: object(1)
memory usage: 691.3+ KB


In [212]:
time_series_data


,Unnamed: 0,ID визита,Пациент,patient_id,Время поступления,День пребывания,chartTime,"tº, C",АД,нАД,...,ВСВЛ,МОК (УЗИ),"tº крови, C","Пульсация лучевой артерии, лев.","Пульсация лучевой артерии, прав.",Проверка CSM,Наполнение капилляров,Парадоксальный пульс,Лодыжечно-плечевой индекс,merge_ids
0,0,17633,Шпакова,60,2025-01-03 15:58:06.3070000,1,2025-01-03 16:04:00,NaN,NaN,71.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60
1,1,17633,Шпакова,60,2025-01-03 15:58:06.3070000,1,2025-01-03 16:05:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60
2,2,17633,Шпакова,60,2025-01-03 15:58:06.3070000,1,2025-01-03 16:10:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60
3,3,17633,Шпакова,60,2025-01-03 15:58:06.3070000,1,2025-01-03 16:15:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60
4,4,17633,Шпакова,60,2025-01-03 15:58:06.3070000,1,2025-01-03 16:20:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43125,43125,19968,Гусак,6322,2025-10-24 13:11:44.2930000,14,2025-11-06 08:06:00,NaN,NaN,171.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6322
43126,43126,19968,Гусак,6322,2025-10-24 13:11:44.2930000,14,2025-11-06 08:30:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6322
43127,43127,19968,Гусак,6322,2025-10-24 13:11:44.2930000,14,2025-11-06 09:00:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6322
43128,43128,19968,Гусак,6322,2025-10-24 13:11:44.2930000,14,2025-11-06 09:06:00,NaN,NaN,158.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6322


### Попытка склеить

Здесь идет попытка склеить два датафрейма по полученному айдишнику

TODO: Из-за того, что в time_series_data находятся данные из датафрейма за 2025 год, не получается собрать ни одного пересечения по id. Для корректной работы требуется собрать со всех загруженных выше датафреймов другие страницы и попробовать фильтровать по ним (другие годы, не 2025)

In [213]:
mergerd_df["patient_id"].apply(lambda x: str(x).split("-")[1])

5       10689
45      11028
59      11164
68      11216
69      11220
        ...  
2825      858
2839      910
2850      947
2853      962
2859      977
Name: patient_id, Length: 4876, dtype: object

In [214]:
mergerd_df["merge_ids"] = mergerd_df["patient_id"].apply(lambda x: str(x).split("-")[1])

In [215]:
time_series_data["merge_ids"] = time_series_data["patient_id"]

In [162]:
ts_valid = time_series_data.merge(mergerd_df[['merge_ids', 'дата поступления', 'дата выписки']], on='merge_ids', how='inner')
# ts_valid = ts_valid[ts_valid['time'] <= ts_valid['target_time']]

In [164]:
ts_valid

,Unnamed: 0,ID визита,Пациент,patient_id,Время поступления,День пребывания,chartTime,"tº, C",АД,нАД,...,"tº крови, C","Пульсация лучевой артерии, лев.","Пульсация лучевой артерии, прав.",Проверка CSM,Наполнение капилляров,Парадоксальный пульс,Лодыжечно-плечевой индекс,merge_ids,дата поступления,дата выписки


------------------------

In [123]:
table_data["дата выписки"] = pd.to_datetime(table_data["дата выписки"], errors='coerce')
table_data["дата поступления"] = pd.to_datetime(table_data["дата поступления"], errors='coerce')

In [ ]:
table_data = table_data.sort_values("дата поступления")

In [32]:
table_data = table_data.sort_values("дата выписки")
time_series_data = time_series_data.sort_values("chartTime")

In [ ]:
ts_valid = time_series_data.merge(table_data[['patient_id', 'target_time']], on='patient_id', how='inner')
ts_valid = ts_valid[ts_valid['time'] <= ts_valid['target_time']]

-----------------------

In [276]:
time_2023_slice = pd.read_excel("./data_raw/time_series/2023.xlsx", sheet_name=None)

In [277]:
for sheet_name, df in time_2023_slice.items():
    print(sheet_name)

Больные
Монитор_1
Монитор_2
Монитор_3
Монитор_4
ИВЛ
Балансы
Лаб
Препараты
ЗППТ


In [278]:
time_2023_slice_1 = time_2023_slice["Монитор_1"]
time_2023_slice_2 = time_2023_slice["ИВЛ"]
time_2023_slice_3 = time_2023_slice["Балансы"]
time_2023_slice_4 = time_2023_slice["Лаб"]
time_2023_slice_5 = time_2023_slice["Препараты"]
time_2022_slice_6 = time_2023_slice["ЗППТ"]

In [279]:
time_2023_slice_1["merge_ids"] = time_2023_slice_1["N иб"]
time_2023_slice_1["merge_ids"] = time_2023_slice_1["merge_ids"].astype('object')

In [267]:
print(mergerd_df[["Код пациента", "merge_ids", "patient_id"]].head(3).to_markdown())

|    | Код пациента   |   merge_ids |   patient_id |
|---:|:---------------|------------:|-------------:|
|  5 | 20-10689       |       10689 |        10689 |
| 45 | 20-11028       |       11028 |        11028 |
| 59 | 20-11164       |       11164 |        11164 |


In [280]:
print(time_2023_slice_1[["merge_ids"]].head(3).to_markdown())

|    |   merge_ids |
|---:|------------:|
|  0 |           1 |
|  1 |           1 |
|  2 |           1 |


In [273]:
mergerd_df["merge_ids"] = mergerd_df["patient_id"]

In [281]:
ts_valid = time_2023_slice_1.merge(mergerd_df[['merge_ids', 'дата поступления', 'дата выписки']], on='merge_ids', how='inner')

In [282]:
ts_valid

,ID визита,Пациент,N иб,Время поступления,День пребывания,chartTime,"tº, C",АД,нАД,ЧСС,...,"tº крови, C","Пульсация лучевой артерии, лев.","Пульсация лучевой артерии, прав.",Проверка CSM,Наполнение капилляров,Парадоксальный пульс,Лодыжечно-плечевой индекс,merge_ids,дата поступления,дата выписки


In [283]:
print(time_2023_slice_1.head(3).to_markdown())

|    |   ID визита | Пациент    |   N иб | Время поступления           |   День пребывания | chartTime                   |   tº, C |   АД |   нАД |   ЧСС |   Пульс |   ЦВД (CVP) |   SpO2 |   ИПД/PPV |   VCO2 |   СВ (CO) |   СИ (CI) |   ДЗЛК |   ДЛА |   НСВ (CCO) |   НСИ (CCI) |   СВ по Фику |   Ударный объем |   ИУО |   ССС |   ИССС |   ЛСС |   ИЛСС |   УРЛЖ |   ИУРЛЖ |   УРПЖ |   ИУРПЖ |   РЛЖ (LCW) |   ИРЛЖ (LCWI) |   Сердечный ритм |   РПЖ |   ИРПЖ |   AaDO2 |   CaO2 |   CvO2 |   Эктопия сердца |   ВГОК |   ВСВЛ |   МОК (УЗИ) |   tº крови, C |   Пульсация лучевой артерии, лев. |   Пульсация лучевой артерии, прав. |   Проверка CSM |   Наполнение капилляров |   Парадоксальный пульс |   Лодыжечно-плечевой индекс |   merge_ids |
|---:|------------:|:-----------|-------:|:----------------------------|------------------:|:----------------------------|--------:|-----:|------:|------:|--------:|------------:|-------:|----------:|-------:|----------:|----------:|-------:|------:|------------

In [227]:
mergerd_df.head(3)

,patient_id,Name,Age,Sex,Наличие в БД,Наличие в файле,STEMI,ЧКВ,Дата STEMI,Вид STEMI,...,HCO3VenMin (a),HCO3VenMin,HCO3VenMax (b),HCO3VenMax (a),HCO3VenMax,BNP (b),BNP (a),BNP,match,merge_ids
5,20-10689,Панченко ЛН,59.0,М,Да,Да,Нет,Да,NaN,NaN,...,23.7,23.7,23.7,23.7,23.7,NaN,NaN,NaN,20-10689,10689
45,20-11028,Марченко ВА,66.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20-11028,11028
59,20-11164,Подленный КИ,73.0,М,Да,Да,Нет,Да,NaN,NaN,...,21.1,21.1,21.1,21.1,21.1,NaN,NaN,NaN,20-11164,11164


### Пересечения по идентификаторам есть, но так как они разных типов, то ничего не мерджится

In [298]:
list_of_time = []
for i in list(time_2023_slice_1["N иб"].unique()):
    list_of_time.append(int(i))

In [296]:
list_of_merge = []
for i in list(mergerd_df["patient_id"].unique()):
    list_of_merge.append(int(i))

In [300]:
for i in list_of_time:
    if i in list_of_merge:
        print("Found!")

Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!
Found!